# Solar Filament V9 — TopoBoost (CPU-only, genuinely new pixel learner)

**What is different:** trains a new **LightGBM multi-scale dark-ridge pixel classifier** from the official *training* polygons, rather than re-training YOLO or changing confidence thresholds. Generates standalone native-resolution filament proposals from the resulting learned probability map, optionally completes short gaps, and compares its instance masks/fusion against your **unchanged V7 0.34 recipe** (when the original V7 saved outputs are attached). The YOLO branch is only a fixed baseline, not the new learner.

**Research inspiration (NOT a claim to reproduce these methods):** Zhou et al., *Masked Vascular Structure Segmentation and Completion in Retinal Images*, **IEEE TMI 2025**, DOI 10.1109/TMI.2025.3538336; GLCP, MICCAI 2025, https://papers.miccai.org/miccai-2025/0376-Paper1868.html; Skeleton Recall Loss, https://arxiv.org/abs/2404.03010. The transfer hypothesis is that dark, thin GONG solar filaments and fine vessels both benefit from multiscale structure evidence and connectivity-aware masks. These methods are not directly applied as pretrained medical networks.

**No GPU, no hidden-label access, no opaque hardcoded test CSV.** V9 trains **only on V4 original train stems**. Tune stays distinct; previously seen audit is marked diagnostic. Any promotion requires a **>=0.02 absolute gain on full tune**, not an invented leaderboard claim. No promise of first place or 0.55.

**Run:** `MODE='TRAIN'` → save `/kaggle/working/solar_topoboost_v9/` as Kaggle Dataset → reattach in another CPU session, `MODE='TUNE'` → `MODE='SUBMIT'`. `MODE='ALL'` possible but CPU test map inference is slower. `MODE='SMOKE'` only tests math/geometry. Save all outputs between sessions.


In [ ]:
import os, json, hashlib, random, warnings, math, time, shutil, sys, subprocess, importlib.util
from pathlib import Path
from collections import defaultdict
from itertools import product
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm
from scipy.optimize import linear_sum_assignment
from IPython.display import display

# --- USER CONFIG ---
MODE = 'TRAIN'            # 'SMOKE' | 'TRAIN' | 'TUNE' | 'SUBMIT' | 'ALL'
SEED=2026; H=W=2048; DEVICE='cpu'
DATA_ROOT=Path('/kaggle/input/datasets/lighittha/solar3k/MAGFiLO_1.0_Kaggle_2026')
SOURCE_V4_ROOT=Path('/kaggle/input/datasets/lighittha/solaroutsp2/solar_cpu_v4')
SOURCE_V7_ROOT=Path('/kaggle/input/datasets/lighittha/solarouts/solar_cpu_v7')
PREVIOUS_V9_ROOT=None  # e.g. Path('/kaggle/input/datasets/.../solar_topoboost_v9')
OUT=Path('/kaggle/working/solar_topoboost_v9' if Path('/kaggle/working').exists() else '/mnt/data/solar_topoboost_v9')
OUT.mkdir(exist_ok=True,parents=True)
# CPU resource budget
MAX_TRAIN_OBSERVATIONS=430      # drawn ONLY from original 513 V4 training observations
TRAIN_CROPS_PER_IMAGE=2
CROP_SIDE=512; TRAIN_POS_PER_CROP=550; TRAIN_HARDNEG_PER_CROP=700; TRAIN_RANDOMNEG_PER_CROP=850
BOOST_TREES=170; BOOST_LEAVES=23
INFER_TILE_SIDE=512; INFER_HALO=48
INFER_PIXEL_GATE=True          # perform LightGBM prediction only on plausible dark-pixel candidates
MAX_INFER_PIXELS_PER_CALL=40000
SCREEN_IMAGES=24; FINALISTS=4; MIN_GAIN=.020
AUTO_INSTALL=True; CLASSICAL_ONLY_WITHOUT_CHECKPOINT=False
# -----------------------------------------------------------
random.seed(SEED);np.random.seed(SEED);cv2.setNumThreads(0)
for package,pipname in [('pycocotools','pycocotools'),('lightgbm','lightgbm>=4.0,<5')]:
    if importlib.util.find_spec(package) is None:
        if not AUTO_INSTALL:raise ImportError(pipname)
        subprocess.check_call([sys.executable,'-m','pip','-q','install',pipname])
import lightgbm as lgb
from pycocotools import mask as coco
print('Mode',MODE,'CPU LightGBM',lgb.__version__,'output',OUT)


## 1. Fixed data provenance and non-overlapping groups

This explicitly preserves the pre-trained V4/V7 training manifest instead of producing new labels from a data split that overlaps their learned weights. Physical GONG images with multiple annotators remain together. The new LightGBM sees only `SPLIT['train']` labels. V7 is optional: if saved caches/selection are unavailable, V9 can be compared with V4 instead, but its baseline may not reproduce the 0.34 submission.

In [ ]:
def sha256(path):
    d=hashlib.sha256()
    with open(path,'rb') as f:
        for buf in iter(lambda:f.read(1024*1024),b''):d.update(buf)
    return d.hexdigest()
def find_data():
    if DATA_ROOT and (DATA_ROOT/'train'/'MAGFiLO_1.0_Annotations_kaggle2026_train.json').is_file():return DATA_ROOT
    options=[p.parent.parent for p in Path('/kaggle/input').rglob('MAGFiLO_1.0_Annotations_kaggle2026_train.json')]
    if len(options)!=1:raise FileNotFoundError('Official data not located uniquely: '+str(options[:5]))
    return options[0]
def find_v4():
    if SOURCE_V4_ROOT and (SOURCE_V4_ROOT/'split_manifest.json').is_file() and (SOURCE_V4_ROOT/'runs/yolo11n-seg/weights/best.pt').is_file():return SOURCE_V4_ROOT
    choices=[p.parent for p in Path('/kaggle/input').rglob('split_manifest.json') if (p.parent/'runs/yolo11n-seg/weights/best.pt').is_file()]
    if len(choices)==1:return choices[0]
    raise FileNotFoundError('Attach the saved V4 YOLO11n best.pt and original split_manifest.json. Matches: '+str(choices[:5]))
DATA=find_data();V4=find_v4()
TRAIN={p.stem:p for p in (DATA/'train'/'train_images').iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')}
TEST={p.stem:p for p in (DATA/'test'/'test_images').iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')}
raw=json.loads((DATA/'train'/'MAGFiLO_1.0_Annotations_kaggle2026_train.json').read_text())
SPLIT=json.loads((V4/'split_manifest.json').read_text())
assert set(SPLIT)=={'train','tune','audit'}
assert set(SPLIT['train'])|set(SPLIT['tune'])|set(SPLIT['audit'])==set(TRAIN)
assert not (set(SPLIT['train'])&set(SPLIT['tune']) or set(SPLIT['train'])&set(SPLIT['audit']) or set(SPLIT['tune'])&set(SPLIT['audit']))
for a in SPLIT:
    for b in SPLIT:
        if a!=b:assert not ({x[:8] for x in SPLIT[a]}&{x[:8] for x in SPLIT[b]})
assert len(TEST)==180 and set(TEST).isdisjoint(TRAIN)
BY_ID=defaultdict(list)
for an in raw['annotations']:BY_ID[str(an['image_id'])].append(an)
OBS=defaultdict(list)
for im in raw['images']:
    stem=Path(im['file_name']).stem
    if stem in TRAIN:OBS[stem].append({'image_id':str(im['id']),'annotations':BY_ID[str(im['id'])]})
assert set(OBS)==set(TRAIN)
V4_WEIGHTS=V4/'runs'/'yolo11n-seg'/'weights'/'best.pt';V4_SHA=sha256(V4_WEIGHTS)
SPLIT_SHA=sha256(V4/'split_manifest.json')
print('Train/tune/previous-audit',*[len(SPLIT[k]) for k in ('train','tune','audit')], 'test',len(TEST))
print('V4 checkpoint SHA',V4_SHA)


## 2. Official training polygon rasterization + multi-annotator medoid selection

The selected training annotation is the version with the highest mean foreground-overlap agreement with other versions. **No test annotations, MAGFiLO label mirrors, or private test data** are loaded. This is a train-label operation only.

In [ ]:
def poly_paths(a):
    seg=a.get('segmentation',[])
    if isinstance(seg,dict) or not seg:return []
    if isinstance(seg[0],(int,float)):seg=[seg]
    out=[]
    for item in seg:
        try:
            p=np.asarray(item,dtype=np.float32).reshape(-1,2)
            if len(p)>=3 and np.isfinite(p).all():out.append(p)
        except (TypeError,ValueError):pass
    return out
def anno_rle(a):
    seg=a.get('segmentation',[])
    if isinstance(seg,dict):
        if isinstance(seg.get('counts'),list):return coco.frPyObjects(seg,H,W)
        return {'size':[H,W],'counts':seg['counts'].encode('ascii') if isinstance(seg['counts'],str) else seg['counts']}
    polygons=[np.clip(p,[0,0],[W-1,H-1]).astype(float).ravel().tolist() for p in poly_paths(a)]
    if not polygons:return None
    r=coco.frPyObjects(polygons,H,W)
    return coco.merge(r) if isinstance(r,list) else r

def union_low(record, side=256):
    mask=np.zeros((side,side),np.uint8)
    for a in record['annotations']:
        for p in poly_paths(a):
            cv2.fillPoly(mask,[np.rint(p*(side/2048)).astype(np.int32)],1)
    return mask>0

def choose_rep(stem):
    versions=OBS[stem]
    if len(versions)==1:return versions[0]
    unions=[union_low(v) for v in versions]
    scores=[]
    for i,a in enumerate(unions):
        vals=[]
        for j,b in enumerate(unions):
            if i==j:continue
            den=a.sum()+b.sum()
            vals.append(2*(a&b).sum()/den if den else 1.0)
        scores.append(float(np.mean(vals)))
    return versions[int(np.argmax(scores))]

def mask_crop(rec,x,y,side):
    mask=np.zeros((side,side),np.uint8)
    for a in rec['annotations']:
        seg=a.get('segmentation',[])
        if isinstance(seg,dict):
            whole=coco.decode(anno_rle(a));mask |= whole[y:y+side,x:x+side].astype(np.uint8)
        else:
            for p in poly_paths(a):
                poly=np.rint(p-[x,y]).astype(np.int32)
                cv2.fillPoly(mask,[poly],1)
    return mask

# Unit test cropped labels: correct foreground and negative pixels
rec={'annotations':[{'segmentation':[[100,100,150,100,150,125,100,125]]}]}
assert mask_crop(rec,64,64,128)[50,60]==1
assert mask_crop(rec,64,64,128)[5,5]==0
print('Annotation mask-crop tests passed')


## 3. New multi-scale dark-tubule features (no deep network)

The representation includes native H-alpha intensity, local contrast at 3 spatial scales, morphological black-hat responses at 2 scales, gradients, 2nd-order ridge curvature and texture. It is trained with **foreground-balanced plus hard-background** pixel sampling. This is a new, supervised learner; standalone filament detections will come from its probability map.

In [ ]:
FEATURE_NAMES=['raw','clahe','dark_gauss_4','dark_gauss_12','dark_gauss_26',
               'blackhat_9','blackhat_25','grad','laplacian','ridge','local_std','smooth_4','texture_diff']

def pixel_features(gray):
    g=np.asarray(gray,dtype=np.uint8)
    f=g.astype(np.float32)/255.0
    clahe=cv2.createCLAHE(clipLimit=1.7,tileGridSize=(8,8)).apply(g)
    c=clahe.astype(np.float32)/255.0
    # Gaussian responses are in native pixels; inference uses a context halo to minimize seams.
    s4=cv2.GaussianBlur(c,(0,0),4)
    s12=cv2.GaussianBlur(c,(0,0),12)
    s26=cv2.GaussianBlur(c,(0,0),26)
    bh9=cv2.morphologyEx(g,cv2.MORPH_BLACKHAT,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(9,9)))
    bh25=cv2.morphologyEx(g,cv2.MORPH_BLACKHAT,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(25,25)))
    dx=cv2.Sobel(s4,cv2.CV_32F,1,0,ksize=3)
    dy=cv2.Sobel(s4,cv2.CV_32F,0,1,ksize=3)
    lap=cv2.Laplacian(s4,cv2.CV_32F,ksize=3)
    dxx=cv2.Sobel(s4,cv2.CV_32F,2,0,ksize=3)
    dxy=cv2.Sobel(s4,cv2.CV_32F,1,1,ksize=3)
    dyy=cv2.Sobel(s4,cv2.CV_32F,0,2,ksize=3)
    delta=np.sqrt(np.maximum(0,(dxx-dyy)**2+4*dxy*dxy))
    ev_hi=(dxx+dyy+delta)/2
    # For dark lines the transverse curvature is locally positive.
    ridge=np.maximum(0,ev_hi)
    sqmean=cv2.GaussianBlur(c*c,(0,0),6)
    local_std=np.sqrt(np.maximum(0,sqmean-cv2.GaussianBlur(c,(0,0),6)**2))
    arr=np.stack([f,c,(s4-c),(s12-c),(s26-c),bh9.astype(np.float32)/64,
                  bh25.astype(np.float32)/64,np.hypot(dx,dy),lap,ridge,
                  local_std,s4,np.abs(s12-s4)],-1).astype(np.float32)
    np.nan_to_num(arr,copy=False,nan=0,posinf=0,neginf=0)
    return np.ascontiguousarray(arr)

sample=np.full((128,128),110,np.uint8)
cv2.line(sample,(15,40),(105,90),45,3)
fmap=pixel_features(sample)
assert fmap.shape==(128,128,len(FEATURE_NAMES)) and np.isfinite(fmap).all()
assert float(fmap[60,60,5])>=0
print('Feature / ridge / finite-value tests passed:',len(FEATURE_NAMES),'channels')


## 4. Train the independent LightGBM filament pixel model

One session on CPU; run `MODE='TRAIN'`. It samples from the original V4 **513 training images only**, stores a `topoboost.txt` model, and writes a provenance file. Save this output as a Kaggle dataset. The feature distribution is deliberately mixed between annotated filament centers, confusing dark pixels near filaments, and random backgrounds.

In [ ]:
MODEL_FILE=OUT/'topoboost.txt'; MODEL_META=OUT/'topoboost_meta.json'
# If previous V9 is mounted, load it without retraining.
def locate_prev9():
    if PREVIOUS_V9_ROOT:
        p=Path(PREVIOUS_V9_ROOT)
        return p if (p/'topoboost.txt').is_file() else None
    options=[p.parent for p in Path('/kaggle/input').rglob('topoboost_meta.json')] if Path('/kaggle/input').exists() else []
    good=[]
    for p in options:
        try:
            m=json.loads((p/'topoboost_meta.json').read_text())
            if m.get('split_sha')==SPLIT_SHA and (p/'topoboost.txt').exists():good.append(p)
        except Exception:pass
    return good[0] if len(good)==1 else None
PREV9=locate_prev9()
if not MODEL_FILE.exists() and PREV9:
    shutil.copy2(PREV9/'topoboost.txt',MODEL_FILE)
    shutil.copy2(PREV9/'topoboost_meta.json',MODEL_META)
    print('Attached pretrained TopoBoost from',PREV9)

def crop_anchor(stem,rec,rng):
    centers=[]
    for a in rec['annotations']:
        for poly in poly_paths(a):
            if len(poly):centers.append(np.mean(np.stack([poly.min(0),poly.max(0)]),axis=0))
    if centers and rng.random()<0.82:
        pt=centers[int(rng.integers(len(centers)))]+rng.normal(0,135,2)
        x=int(np.clip(pt[0]-CROP_SIDE/2,0,W-CROP_SIDE))
        y=int(np.clip(pt[1]-CROP_SIDE/2,0,H-CROP_SIDE))
    else:
        x=int(rng.integers(W-CROP_SIDE+1));y=int(rng.integers(H-CROP_SIDE+1))
    return x,y

def draw_train_rows(stem,rng):
    image=cv2.imread(str(TRAIN[stem]),cv2.IMREAD_GRAYSCALE)
    if image is None or image.shape!=(H,W):raise RuntimeError('Bad training image: '+stem)
    rec=choose_rep(stem)
    arrays=[];classes=[]
    for _ in range(TRAIN_CROPS_PER_IMAGE):
        x,y=crop_anchor(stem,rec,rng)
        img=image[y:y+CROP_SIDE,x:x+CROP_SIDE]
        mask=mask_crop(rec,x,y,CROP_SIDE)
        feat=pixel_features(img).reshape(-1,len(FEATURE_NAMES))
        yy=mask.reshape(-1)
        pos=np.flatnonzero(yy>0)
        if len(pos):
            i=rng.choice(pos,min(TRAIN_POS_PER_CROP,len(pos)),replace=False)
            arrays.append(feat[i]);classes.append(np.ones(len(i),np.uint8))
        near=cv2.dilate(mask,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(37,37)))
        hard=np.flatnonzero(((near>0)&(mask==0)).ravel())
        if len(hard):
            i=rng.choice(hard,min(TRAIN_HARDNEG_PER_CROP,len(hard)),replace=False)
            arrays.append(feat[i]);classes.append(np.zeros(len(i),np.uint8))
        background=np.flatnonzero((yy==0)&(img.ravel()>12))
        if len(background):
            i=rng.choice(background,min(TRAIN_RANDOMNEG_PER_CROP,len(background)),replace=False)
            arrays.append(feat[i]);classes.append(np.zeros(len(i),np.uint8))
    return (np.concatenate(arrays),np.concatenate(classes)) if arrays else (np.empty((0,len(FEATURE_NAMES)),np.float32),np.empty(0,np.uint8))

def train_topoboost():
    if MODEL_FILE.is_file() and MODEL_META.is_file():
        print('Using existing model:',MODEL_FILE);return lgb.Booster(model_file=str(MODEL_FILE))
    rng=np.random.default_rng(SEED+904)
    stems=np.array(sorted(SPLIT['train']))
    rng.shuffle(stems)
    stems=stems[:min(MAX_TRAIN_OBSERVATIONS,len(stems))]
    # None of the original 97 tune / 97 audit observations are used in any supervised fit.
    X=[];Y=[];t0=time.time()
    for stem in tqdm(stems,desc='CPU supervised native H-alpha feature extraction'):
        a,b=draw_train_rows(stem,rng)
        if len(b):X.append(a);Y.append(b)
    if not X:raise RuntimeError('No supervised train pixels: verify official polygons')
    X=np.concatenate(X);Y=np.concatenate(Y)
    print('LightGBM training rows',len(Y),'positive fraction after sampling',float(Y.mean()))
    assert 0<Y.mean()<1
    pack=lgb.Dataset(X,label=Y,feature_name=FEATURE_NAMES,free_raw_data=True)
    params={'objective':'binary','metric':'auc','verbosity':-1,'learning_rate':0.055,
            'num_leaves':BOOST_LEAVES,'min_data_in_leaf':95,'feature_fraction':.90,
            'bagging_fraction':.85,'bagging_freq':1,'lambda_l2':3.0,'num_threads':min(4,os.cpu_count() or 2),'seed':SEED}
    booster=lgb.train(params,pack,num_boost_round=BOOST_TREES)
    booster.save_model(str(MODEL_FILE))
    meta={'model':'CPU native-scale multiscale-ridge LightGBM','feature_names':FEATURE_NAMES,
          'split_sha':SPLIT_SHA,'v4_model_sha':V4_SHA,'train_only_stems':list(map(str,stems)),
          'n_samples':int(len(Y)),'sampled_positive_fraction':float(Y.mean()),
          'model_sha256':sha256(MODEL_FILE),'seconds':round(time.time()-t0,1),
          'no_test_labels':True,'no_embedded_test_predictions':True}
    MODEL_META.write_text(json.dumps(meta,indent=2))
    del X,Y
    print('Trained new TopoBoost native-pixel model:',MODEL_FILE,meta['model_sha256'])
    return booster
BOOSTER=train_topoboost() if MODE in ('TRAIN','ALL') else (lgb.Booster(model_file=str(MODEL_FILE)) if MODEL_FILE.exists() else None)
if MODE in ('TUNE','SUBMIT') and BOOSTER is None:raise FileNotFoundError('Run TRAIN first and attach its saved output dataset')


## 5. CPU prediction with native 2048 resolution and safe caching

Each 512² tile is processed with a 48-pixel context halo, converted into the same 13 features as training. A conservative *dark candidate gate* avoids scoring clearly non-filament pixels; this is an efficiency heuristic, not test-label information. Caches are keyed to model SHA, feature version, gate and image size. A saved full-resolution probability map is compressed to 8-bit PNG and survives session restarts.

In [ ]:
PRED=OUT/'prob_maps';PRED.mkdir(exist_ok=True)
PIXEL_MODEL_SHA=sha256(MODEL_FILE) if MODEL_FILE.is_file() else None
PRED_CONFIG={'boost_sha':PIXEL_MODEL_SHA,'side':INFER_TILE_SIDE,'halo':INFER_HALO,
             'gate':INFER_PIXEL_GATE,'version':'v9-lightgbm-ridge-v2'}

def prob_paths(stem):return PRED/(stem+'.png'),PRED/(stem+'.json')
def prev_prob_paths(stem):
    if PREV9:return PREV9/'prob_maps'/(stem+'.png'),PREV9/'prob_maps'/(stem+'.json')
    return None,None

def read_prob(stem):
    options=[prob_paths(stem),prev_prob_paths(stem)]
    for pic,meta in options:
        if pic is None or not pic.exists() or not meta.exists():continue
        try:
            contract=json.loads(meta.read_text())
            if contract!=dict(PRED_CONFIG,stem=stem):continue
            im=cv2.imread(str(pic),cv2.IMREAD_GRAYSCALE)
            if im is not None and im.shape==(H,W):return im
        except Exception:pass
    return None

def predict_probs(image,booster):
    if image.shape!=(H,W):raise ValueError('Expected native 2048x2048 H-alpha')
    out=np.zeros((H,W),np.uint8)
    step=INFER_TILE_SIDE;pad=INFER_HALO
    for y in range(0,H,step):
        for x in range(0,W,step):
            y1=min(y+step,H);x1=min(x+step,W)
            ya=max(0,y-pad);yb=min(H,y1+pad);xa=max(0,x-pad);xb=min(W,x1+pad)
            crop=image[ya:yb,xa:xb]
            ff=pixel_features(crop)[y-ya:y1-ya,x-xa:x1-xa]
            flat=ff.reshape(-1,len(FEATURE_NAMES))
            if INFER_PIXEL_GATE:
                # High recall gate: any significant local darkness, strong black-hat, or dark native intensity.
                eligible=((flat[:,3]>0.010)|(flat[:,4]>0.014)|
                          (flat[:,5]>0.050)|(flat[:,6]>0.070)|
                          ((flat[:,0]<.25)&(flat[:,0]>.045)))
            else:eligible=np.ones(len(flat),dtype=bool)
            indices=np.flatnonzero(eligible)
            pp=np.zeros(len(flat),np.uint8)
            for k in range(0,len(indices),MAX_INFER_PIXELS_PER_CALL):
                sel=indices[k:k+MAX_INFER_PIXELS_PER_CALL]
                score=booster.predict(flat[sel],num_threads=min(4,os.cpu_count() or 2))
                pp[sel]=np.uint8(np.clip(np.rint(score*255),0,255))
            out[y:y1,x:x1]=pp.reshape(y1-y,x1-x)
    return out

def get_prob(stem,imgs):
    arr=read_prob(stem)
    if arr is not None:return arr
    if BOOSTER is None:raise RuntimeError('Train/reload TopoBoost before inference')
    im=cv2.imread(str(imgs[stem]),cv2.IMREAD_GRAYSCALE)
    if im is None or im.shape!=(H,W):raise RuntimeError('Missing/incorrect image '+stem)
    arr=predict_probs(im,BOOSTER)
    p,m=prob_paths(stem)
    if not cv2.imwrite(str(p),arr,[cv2.IMWRITE_PNG_COMPRESSION,3]):raise IOError(str(p))
    m.write_text(json.dumps(dict(PRED_CONFIG,stem=stem),sort_keys=True))
    check=cv2.imread(str(p),cv2.IMREAD_GRAYSCALE)
    assert check is not None and np.array_equal(check,arr)
    return arr

# Test save/load compression and seam-safe area coverage without running the 2048 model
im=np.zeros((72,72),np.uint8);im[7:15,22:65]=128
ok,buf=cv2.imencode('.png',im)
assert ok and np.array_equal(cv2.imdecode(buf,cv2.IMREAD_GRAYSCALE),im)
assert sum((min(y+INFER_TILE_SIDE,H)-y)*(min(x+INFER_TILE_SIDE,W)-x)
            for y in range(0,H,INFER_TILE_SIDE) for x in range(0,W,INFER_TILE_SIDE))==H*W
print('Prediction serialization + complete-image tiling tests passed')


## 6. New instance discovery + topology-inspired short-gap completion

Predict an *independent* set of instances from the learned map via connected components. A 3×3 or 5×5 closing can join short breaks but may accidentally merge separate filaments: **both modes are tuned separately against full instance PQ**. We neither optimize nor train on held-out test labels. This is a light CPU adaptation motivated by MaskVSC/GLCP connectivity ideas, not their exact implementation.

In [ ]:
def encode(m):return coco.encode(np.asfortranarray(np.asarray(m,dtype=np.uint8)))
def rle_from_counts(counts):
    if isinstance(counts,dict):
        return {'size':[H,W],'counts':counts['counts'].encode('ascii') if isinstance(counts['counts'],str) else counts['counts']}
    return {'size':[H,W],'counts':counts.encode('ascii') if isinstance(counts,str) else counts}
def ascii_rle(r):return r['counts'].decode('ascii') if isinstance(r['counts'],bytes) else str(r['counts'])
def disjoint(rles):
    if len(rles)<2:return True
    mat=np.asarray(coco.iou(rles,rles,[0]*len(rles)))
    np.fill_diagonal(mat,0)
    return bool(np.all(mat==0))

INSTANCE_DIR=OUT/'boost_instances';INSTANCE_DIR.mkdir(exist_ok=True)
def standalone_candidates(stem,images,threshold=.65,min_area=350,close=3):
    token=f'{stem}__t{int(threshold*1000)}_a{min_area}_c{close}.json'
    file=INSTANCE_DIR/token
    prev=PREV9/'boost_instances'/token if PREV9 else None
    for p in (file,prev):
        if p is None or not p.exists():continue
        try:
            d=json.loads(p.read_text())
            if d['model_sha']==PIXEL_MODEL_SHA and d['options']==[threshold,min_area,close]:return d['instances']
        except Exception:pass
    prob=get_prob(stem,images)
    binary=np.uint8(prob>=int(threshold*255))
    if close>1:binary=cv2.morphologyEx(binary,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(close,close)))
    n,lab,stats,_=cv2.connectedComponentsWithStats(binary,8)
    result=[]
    for i in range(1,n):
        x,y,w,h,area=map(int,stats[i])
        if area<min_area or area>110000 or max(w,h)<25:continue
        local=(lab[y:y+h,x:x+w]==i)
        # Compact dark spots are less filament-like than elongated ridges; avoid hard filtering curved objects.
        fill=area/max(1,w*h)
        if fill>.72 and min(w,h)>28:continue
        mean_score=float(prob[y:y+h,x:x+w][local].mean()/255)
        if mean_score<threshold+.025:continue
        whole=np.zeros((H,W),np.uint8);whole[y:y+h,x:x+w]=local.astype(np.uint8)
        rr=encode(whole)
        result.append({'counts':ascii_rle(rr),'score':round(mean_score,6),'area':int(area),'kind':'topoboost'})
    file.write_text(json.dumps({'model_sha':PIXEL_MODEL_SHA,'options':[threshold,min_area,close],'instances':result},separators=(',',':')))
    return result

def exclusive_ranked(candidates,min_area=350,dup=.50,containment=.88):
    if not candidates:return []
    occupied=np.zeros((H,W),np.uint8);used=[];out=[]
    for p in sorted(candidates,key=lambda v:(-v['score'],-v['area'])):
        if p['area']<min_area:continue
        rr=rle_from_counts(p['counts'])
        if used:
            ious=np.asarray(coco.iou([rr],used,[0]*len(used)))[0]
            prev_area=np.asarray(coco.area(used),dtype=np.float64)
            intersection=ious*(prev_area+p['area'])/(1+ious+1e-8)
            if np.any(ious>=dup) or np.any(intersection/max(1,p['area'])>=containment):continue
        mm=coco.decode(rr).astype(bool)&(occupied==0)
        if mm.sum()<min_area:continue
        occupied[mm]=1;used.append(rr);out.append(encode(mm))
    assert disjoint(out)
    return out

# Toy: two overlapping native filament structures cannot produce overlapping output.
a=np.zeros((H,W),np.uint8);b=np.zeros_like(a)
a[10:60,10:80]=1;b[30:100,35:105]=1
rr1,rr2=encode(a),encode(b)
assert not disjoint([rr1,rr2])
toy=exclusive_ranked([{'counts':ascii_rle(rr1),'score':.91,'area':int(a.sum())},
                      {'counts':ascii_rle(rr2),'score':.85,'area':int(b.sum())}],min_area=80)
assert disjoint(toy)
del a,b
print('Instance connected-component/RLE deduplication tests passed')


## 7. Faithful V4 / V7 reference (unchanged)

Reads or regenerates legitimate predictions from your saved V4 YOLO checkpoint. If your saved V7 selection and tiles are accessible, reproduces its algorithm exactly; otherwise use the original V4 baseline and clearly report that the 0.34 V7 reference was not reproduced. **V9's booster can still learn and generate its own instances with no GPU.**

In [ ]:
V7=None;V7CFG=None;V7_SELECTION=None
if SOURCE_V7_ROOT and (SOURCE_V7_ROOT/'v7_config.json').exists():V7=SOURCE_V7_ROOT
else:
    options=[p.parent for p in Path('/kaggle/input').rglob('v7_config.json')] if Path('/kaggle/input').exists() else []
    matches=[]
    for p in options:
        try:
            d=json.loads((p/'v7_config.json').read_text())
            if d.get('checkpoint_sha256')==V4_SHA and d.get('split_sha256')==SPLIT_SHA:matches.append(p)
        except Exception:pass
    V7=matches[0] if len(matches)==1 else None
if V7:
    V7CFG=json.loads((V7/'v7_config.json').read_text())
    if V7CFG.get('checkpoint_sha256')!=V4_SHA or V7CFG.get('split_sha256')!=SPLIT_SHA:V7=None
    elif (V7/'selection.json').exists():V7_SELECTION=json.loads((V7/'selection.json').read_text()).get('strategy')
print('Saved V7 baseline',V7,'recipe',V7_SELECTION)
BASE_DIR=OUT/'base_proposals';BASE_DIR.mkdir(exist_ok=True)
BASE_MODEL=None

def get_base_model():
    global BASE_MODEL
    if BASE_MODEL is None:
        if importlib.util.find_spec('ultralytics') is None:
            if not AUTO_INSTALL:raise ImportError('Install ultralytics to regenerate baseline predictions')
            subprocess.check_call([sys.executable,'-m','pip','-q','install','ultralytics>=8.3,<9'])
        from ultralytics import YOLO
        BASE_MODEL=YOLO(str(V4_WEIGHTS))
    return BASE_MODEL

def model_infer_props(gray,kind='global',x=0,y=0,side=1280):
    import torch
    with torch.inference_mode():
        image=cv2.cvtColor(gray,cv2.COLOR_GRAY2BGR)
        result=get_base_model().predict(source=image,imgsz=1024,conf=.10,iou=.82,retina_masks=True,
                                         agnostic_nms=True,max_det=65,device='cpu',verbose=False)[0]
    if result.masks is None:return []
    scores=result.boxes.conf.detach().cpu().numpy()
    arr=result.masks.data.detach().cpu().numpy()
    props=[]
    for confidence,m in zip(scores,arr):
        patch=np.uint8(m>.5)
        if patch.shape!=gray.shape:patch=cv2.resize(patch,(gray.shape[1],gray.shape[0]),interpolation=cv2.INTER_NEAREST)
        if not patch.any():continue
        if kind=='tile':
            edge=bool((x>0 and patch[:,:12].any()) or (y>0 and patch[:12,:].any()) or
                      (x+side<W and patch[:,-12:].any()) or (y+side<H and patch[-12:,:].any()))
            full=np.zeros((H,W),np.uint8);full[y:y+side,x:x+side]=patch
        else:full=patch;edge=False
        rr=encode(full)
        props.append({'counts':ascii_rle(rr),'score':float(confidence),'area':int(coco.area(rr)),
                      'origin':kind,'truncated':edge})
    return props

def existing_v4_cache(stem):
    f=V4/'cache'/'yolo11n-seg'/(stem+'.json')
    if not f.is_file():return None
    try:
        d=json.loads(f.read_text())
        if (d.get('stem')==stem and d.get('checkpoint_sha256')==V4_SHA and d.get('predict_imgsz')==1024
                and d.get('tta') is False and d.get('source')=='LIVE_IMAGE_INFERENCE'):return d['proposals']
    except Exception:pass
    return None

def get_global(stem,images):
    file=BASE_DIR/(stem+'.json')
    for path in (file,(PREV9/'base_proposals'/(stem+'.json')) if PREV9 else None):
        if path and path.exists():
            try:
                d=json.loads(path.read_text())
                if d.get('model_sha')==V4_SHA and d.get('stem')==stem:return d['proposals']
            except Exception:pass
    cached=existing_v4_cache(stem)
    if cached is not None:return cached
    img=cv2.imread(str(images[stem]),cv2.IMREAD_GRAYSCALE)
    if img is None or img.shape!=(H,W):raise RuntimeError(stem)
    result=model_infer_props(img)
    file.write_text(json.dumps({'stem':stem,'model_sha':V4_SHA,'proposals':result}))
    return result

def get_v7_tiles(stem,images):
    if not V7:return None
    f=V7/'tiles'/(stem+'.json')
    if f.is_file():
        try:
            d=json.loads(f.read_text())
            if (d.get('stem')==stem and d.get('source')=='LIVE_TEST_OR_TRAIN_JPEG_INFERENCE'
                and d.get('kind')=='tile' and all(d.get(k)==v for k,v in V7CFG.items())):return d['proposals']
        except Exception:pass
    # Reconstruct V7 tiles from the original trained V4 checkpoint when caches are absent.
    cache=BASE_DIR/(stem+'_v7tiles.json')
    if cache.exists():
        try:
            d=json.loads(cache.read_text())
            if d.get('model_sha')==V4_SHA and d.get('v7config')==V7CFG:return d['proposals']
        except Exception:pass
    im=cv2.imread(str(images[stem]),cv2.IMREAD_GRAYSCALE)
    if im is None:raise RuntimeError(stem)
    side=int(V7CFG.get('tile_side',1280))
    windows=[(0,0),(W-side,0),(0,H-side),(W-side,H-side)]
    if V7CFG.get('tile_scheme')=='FIVE':windows.append(((W-side)//2,(H-side)//2))
    found=[]
    for x,y in sorted(set(windows)):
        found.extend(model_infer_props(im[y:y+side,x:x+side],kind='tile',x=x,y=y,side=side))
    cache.write_text(json.dumps({'model_sha':V4_SHA,'v7config':V7CFG,'proposals':found}))
    return found

def v4_baseline(stem,images):
    props=get_global(stem,images)
    ranked=sorted([p for p in props if p['score']>=.32 and p['area']>=350],key=lambda p:(-p['score'],-p['area']))
    occupied=np.zeros((H,W),np.uint8);prev=[];out=[]
    for p in ranked:
        r=rle_from_counts(p['counts'])
        if prev:
            iou=np.asarray(coco.iou([r],prev,[0]*len(prev)))[0]
            if iou.size and np.max(iou)>=.5:continue
        prev.append(r)
        mm=coco.decode(r).astype(bool)&(occupied==0)
        if mm.sum()<350:continue
        occupied[mm]=1;out.append(encode(mm))
    return out

def v7_baseline(stem,images):
    if V7_SELECTION is None:return v4_baseline(stem,images)
    if V7_SELECTION=='v4':return v4_baseline(stem,images)
    cfg=V7_SELECTION
    if not isinstance(cfg,dict):raise RuntimeError('Unexpected old V7 recipe format')
    gg=[dict(p,origin='global',truncated=False) for p in get_global(stem,images)]
    tt=get_v7_tiles(stem,images) if cfg['mode']!='global' else []
    if tt is None:raise RuntimeError('V7 tile source unavailable')
    kept=[]
    for p in gg+tt:
        is_tile=p['origin']=='tile'
        if p['score']<(cfg['tile_conf'] if is_tile else cfg['global_conf']):continue
        if p['area']<cfg['min_area'] or (is_tile and cfg['seam_guard'] and p.get('truncated',False)):continue
        kept.append(dict(p,rank=p['score']+(cfg['tile_bonus'] if is_tile else 0)))
    kept.sort(key=lambda q:(-q['rank'],-q['area']))
    occ=np.zeros((H,W),np.uint8);prior=[];out=[]
    for p in kept:
        r=rle_from_counts(p['counts'])
        if prior:
            ious=np.asarray(coco.iou([r],prior,[0]*len(prior)))[0]
            areas=np.asarray(coco.area(prior))
            ints=ious*(areas+p['area'])/(1+ious+1e-9)
            if np.any(ious>=cfg['nms_iou']) or np.any(ints/max(1,p['area'])>=cfg['containment']):continue
        m=coco.decode(r).astype(bool)&(occ==0)
        if m.sum()<cfg['min_area'] or m.sum()/max(1,p['area'])<cfg['min_free_fraction']:continue
        occ[m]=1;prior.append(r);out.append(encode(m))
    assert disjoint(out)
    return out

def baseline(stem,images):return v7_baseline(stem,images) if V7_SELECTION is not None else v4_baseline(stem,images)
print('Baseline available:', 'V7 recipe' if V7_SELECTION is not None else 'V4 ONLY (0.34 not reproduced)')


## 8. New baseline + TopoBoost hybrid candidates

We compare `boost_only` (independent model), `rescue` (new filament detections where baseline missed), and `guided` (small local corrections to existing masks). All masks must be disjoint. **Baseline masks are always preserved as a candidate.** Promotion requires full-tune gain. The potential new discovery comes from a separately trained LightGBM model, not from the old YOLO output.

In [ ]:
def guided_baseline(stem,images,base,params):
    prob=get_prob(stem,images)
    occ=np.zeros((H,W),np.uint8);out=[]
    threshold=int(params['pixel_threshold']*255)
    for rr in base:
        raw=coco.decode(rr).astype(np.uint8)
        ys,xs=np.where(raw)
        if len(ys)==0:continue
        x0=max(0,int(xs.min())-12);x1=min(W,int(xs.max())+13)
        y0=max(0,int(ys.min())-12);y1=min(H,int(ys.max())+13)
        roi=raw[y0:y1,x0:x1]
        dil=cv2.dilate(roi,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(5,5)))>0
        high=prob[y0:y1,x0:x1]>=threshold
        corrected=(roi>0)&(prob[y0:y1,x0:x1]>=max(0,threshold-40))
        corrected |= dil & high
        ratio=corrected.sum()/max(1,roi.sum())
        agreement=np.logical_and(corrected,roi>0).sum()/max(1,np.logical_or(corrected,roi>0).sum())
        if not (.82<=ratio<=1.18 and agreement>.82):corrected=roi>0
        clear=corrected & (occ[y0:y1,x0:x1]==0)
        if clear.sum()<70:continue
        mm=np.zeros((H,W),np.uint8);mm[y0:y1,x0:x1]=clear
        occ[y0:y1,x0:x1][clear]=1
        out.append(encode(mm))
    return out

def recipe(stem,images,params):
    if params=='baseline':return baseline(stem,images)
    mode=params['mode'];thr=params['pixel_threshold'];area=params['min_area'];closing=params['closing']
    base=baseline(stem,images) if mode!='boost_only' else []
    if mode=='guided':return guided_baseline(stem,images,base,params)
    items=standalone_candidates(stem,images,threshold=thr,min_area=area,close=closing)
    if mode=='boost_only':return exclusive_ranked(items,min_area=area)
    # Rescue: baseline survives intact. Drop learned candidates if they strongly overlap or damage earlier masks.
    occupied=np.zeros((H,W),np.uint8)
    for mask in base:occupied|=coco.decode(mask).astype(np.uint8)
    result=list(base)
    for obj in sorted(items,key=lambda x:-x['score']):
        rr=rle_from_counts(obj['counts'])
        whole=coco.decode(rr).astype(bool)
        n=whole.sum()
        if n==0:continue
        fraction=float((whole & (occupied>0)).sum()/n)
        if fraction>params['max_overlap']:continue
        free=whole & (occupied==0)
        if free.sum()<area or free.sum()/n<.70:continue
        occupied[free]=1
        result.append(encode(free))
    assert disjoint(result)
    return result


## 9. Per-annotator instance PQ (one-to-one) + transparent error accounting

Each annotation version is evaluated separately. A Hungarian assignment provides one-to-one instance matches at IoU > 0.5. **The organizer's public evaluator remains authoritative**; measure both pooled and per-annotation means. High scores from overlapping or multi-match cheats are explicitly disallowed.

In [ ]:
GT_CACHE={}
def truth(stem):
    if stem not in GT_CACHE:
        GT_CACHE[stem]=[]
        for v in OBS[stem]:
            GT_CACHE[stem].append((v['image_id'],[r for a in v['annotations'] if (r:=anno_rle(a)) is not None]))
    return GT_CACHE[stem]

def pair_counts(gt,pr):
    if not gt and not pr:return np.zeros(4,float)
    if not gt:return np.array([0,0,len(pr),0],float)
    if not pr:return np.array([0,0,0,len(gt)],float)
    iou=np.asarray(coco.iou(gt,pr,[0]*len(pr)),dtype=float)
    row,col=linear_sum_assignment(-iou)
    keep=iou[row,col]>.5
    tp=int(keep.sum())
    return np.array([iou[row[keep],col[keep]].sum(),tp,len(pr)-tp,len(gt)-tp],float)

def describe(v):
    si,tp,fp,fn=map(float,v)
    den=tp+.5*(fp+fn)
    return {'pq':float(si/den) if den else 0.,'sq':float(si/tp) if tp else 0.,
            'rq':float(tp/den) if den else 0.,'tp':int(tp),'fp':int(fp),'fn':int(fn)}

def evaluate(params,stems,save_detail=False):
    total=np.zeros(4,float);rows=[]
    for stem in stems:
        pr=recipe(stem,TRAIN,params)
        for version,gt in truth(stem):
            counts=pair_counts(gt,pr)
            total+=counts
            if save_detail:rows.append({'stem':stem,'annotation':version,**describe(counts)})
    return describe(total),pd.DataFrame(rows)

zz=np.zeros((H,W),np.uint8);zz[12:44,30:65]=1
r=encode(zz)
assert abs(describe(pair_counts([r],[r]))['pq']-1)<1e-8
assert describe(pair_counts([r],[]))['fn']==1
assert describe(pair_counts([],[r]))['fp']==1
print('Instance PQ and RLE unit tests passed')


## 10. Screening → entire 97-image tune → historical audit; conservative selection

Screens 24 tune observations, evaluates top finalists on **all 97 physical tune observations**, and does not promote a new method unless it wins by >=0.020 absolute PQ. The 97-image old audit was seen in prior sessions and is marked historical, not a pristine test. This does not optimize against Kaggle public test labels/scores. Training and evaluation are fully separable and cache-resumable.

In [ ]:
SELECT=OUT/'selection.json'; SCREEN_CSV=OUT/'screening_results.csv'; FULL_CSV=OUT/'full_tune_results.csv'
A_PATH=OUT/'audit_comparison.json'
TUNE_STEMS=sorted(SPLIT['tune']);rng=random.Random(SEED+916);rng.shuffle(TUNE_STEMS)
FAST_TUNE=sorted(TUNE_STEMS[:SCREEN_IMAGES]);ALL_TUNE=sorted(SPLIT['tune'])
# Need a new learned proposal branch, not endless confidence tuning.
PARAMS=[]
for thresh,area,closing in product([.48,.63,.76],[220,350],[1,3]):
    PARAMS.append({'mode':'boost_only','pixel_threshold':thresh,'min_area':area,'closing':closing,'max_overlap':0.})
    for overlap in [0.15]:
        PARAMS.append({'mode':'rescue','pixel_threshold':thresh,'min_area':area,'closing':closing,'max_overlap':overlap})
for thresh in [.55,.70]:
    PARAMS.append({'mode':'guided','pixel_threshold':thresh,'min_area':350,'closing':1,'max_overlap':0.})

def build_prob_maps(stems,images):
    for stem in tqdm(stems,desc='CPU TopoBoost native probability maps'):
        if read_prob(stem) is None:get_prob(stem,images)

def run_tune():
    build_prob_maps(FAST_TUNE,TRAIN)
    original,_=evaluate('baseline',FAST_TUNE)
    records=[{'index':-1,'method':'baseline','stage':'screen',**original}]
    for idx,config in enumerate(tqdm(PARAMS,desc='TopoBoost CPU candidate screen')):
        metrics,_=evaluate(config,FAST_TUNE)
        records.append({'index':idx,'method':config['mode'],'stage':'screen',**metrics})
    board=pd.DataFrame(records).sort_values(['pq','rq'],ascending=False)
    board.to_csv(SCREEN_CSV,index=False);display(board.head(12))
    finalist_idx=[int(i) for i in board['index'] if int(i)>=0][:FINALISTS]
    build_prob_maps(ALL_TUNE,TRAIN)
    old,_=evaluate('baseline',ALL_TUNE)
    full=[{'index':-1,'method':'baseline',**old}]
    for idx in tqdm(finalist_idx,desc='Full original 97-image grouped tune'):
        metrics,_=evaluate(PARAMS[idx],ALL_TUNE)
        full.append({'index':idx,'method':PARAMS[idx]['mode'],**metrics})
    final=pd.DataFrame(full).sort_values(['pq','rq'],ascending=False)
    final.to_csv(FULL_CSV,index=False);display(final)
    top=final.iloc[0]
    gain=float(top['pq']-old['pq'])
    winner='baseline' if int(top['index'])<0 or gain<MIN_GAIN else PARAMS[int(top['index'])]
    seal={'choice':winner,'reported_best_tune':float(top['pq']),
          'baseline_tune':float(old['pq']),'gain':gain,'promotion_min_gain':MIN_GAIN,
          'split_sha':SPLIT_SHA,'v4_sha':V4_SHA,'boost_sha':PIXEL_MODEL_SHA,
          'v7_recipe_available':V7_SELECTION is not None,'tune_stems':ALL_TUNE,
          'no_test_labels_used':True,'comparison_protocol':'24-screen then full 97-tune; historical audit not used for selection'}
    SELECT.write_text(json.dumps(seal,indent=2))
    print('FROZEN SELECTED:',winner,'baseline full tune',old['pq'],'gain',gain)
    # previously inspected audit: only diagnostic; no retuning
    audit=sorted(SPLIT['audit'])
    if winner!='baseline':build_prob_maps(audit,TRAIN)
    old_a,_=evaluate('baseline',audit)
    new_a,_=evaluate(winner,audit)
    report={'status':'HISTORICAL_AUDIT_ALREADY_SEEN','baseline':old_a,'new':new_a,
            'delta_pq':new_a['pq']-old_a['pq'],'winner':winner}
    A_PATH.write_text(json.dumps(report,indent=2))
    print('Old audit diagnostic:',json.dumps(report,indent=2))
    return seal

if MODE in ('TUNE','ALL'):
    selected=run_tune()
elif SELECT.exists():selected=json.loads(SELECT.read_text())
elif PREV9 and (PREV9/'selection.json').exists():
    selected=json.loads((PREV9/'selection.json').read_text());shutil.copy2(PREV9/'selection.json',SELECT)
else:selected=None


## 11. Live test inference, checked disjoint instance RLE, and final CSV

The selected method determines test inference. **If it is baseline, no booster candidate is inserted**. Every prediction is produced from its corresponding H-alpha image using verified model files. No dummy instances and no embedded RLE submission. Exports checkpoint + result hashes and counts of filament instances/image.

In [ ]:
def save_csv():
    if selected is None:raise RuntimeError('First run MODE=TUNE with saved TopoBoost model, or attach previous V9 output')
    assert selected['split_sha']==SPLIT_SHA and selected['v4_sha']==V4_SHA and selected['boost_sha']==PIXEL_MODEL_SHA
    choice=selected['choice']
    if choice!='baseline':build_prob_maps(sorted(TEST),TEST)
    records=[];stats=[]
    for stem in tqdm(sorted(TEST),desc='Final live native-image inference'):
        if choice=='baseline':
            masks=baseline(stem,TEST)
        else:
            # recipe's image lookup must be TEST, not TRAIN
            masks=recipe(stem,TEST,choice)
        assert disjoint(masks),f'Overlapping instances in {stem}'
        for j,rr in enumerate(masks,1):
            assert int(coco.area(rr))>0
            records.append({'filament_id':f'{stem}_{j}','segmentation_rle':ascii_rle(rr)})
        stats.append({'stem':stem,'instances':len(masks)})
    df=pd.DataFrame(records,columns=['filament_id','segmentation_rle'])
    assert not df.empty and df['filament_id'].is_unique and not df.isna().any().any()
    dst=Path('/kaggle/working/submission.csv' if Path('/kaggle/working').exists() else OUT/'submission.csv')
    df.to_csv(dst,index=False)
    # Read CSV back, decode compressed RLE and assert disjointness for every physical image.
    check=pd.read_csv(dst,keep_default_na=False)
    assert len(check)==len(df) and set(check.columns)=={'filament_id','segmentation_rle'}
    for s,group in check.groupby(check.filament_id.str.rsplit('_',n=1).str[0]):
        assert s in TEST
        rr=[rle_from_counts(c) for c in group['segmentation_rle']]
        assert all(int(coco.area(r))>0 for r in rr) and disjoint(rr)
    stat=pd.DataFrame(stats);stat.to_csv(OUT/'test_instance_counts.csv',index=False)
    report={'model':'TopoBoost native multiscale ridge LightGBM + optional validated rescue',
            'chosen':choice,'rows':int(len(df)),'test_images':len(TEST),
            'images_with_instances':int((stat.instances>0).sum()),
            'submission_sha256':sha256(dst),'topoboost_sha256':PIXEL_MODEL_SHA,
            'v4_sha256':V4_SHA,'baseline_v7_attached':V7_SELECTION is not None,
            'all_masks_from_live_image_inference':True,'test_labels_used':False}
    (OUT/'submission_report.json').write_text(json.dumps(report,indent=2))
    print('Generated:',dst,'\n',json.dumps(report,indent=2))

if MODE in ('SUBMIT','ALL'):save_csv()


## 12. Ablations and interpretation

Review `screening_results.csv`, `full_tune_results.csv`, `audit_comparison.json` and `submission_report.json`. A gain smaller than 0.020 is deliberately considered insufficient to promote a modification. **If this new LightGBM cannot improve grouped-validation PQ, do not spend another leaderboard attempt on it.**

If the new model fails: inspect false-positive masks and label disagreement first. The organizer states that **any PQ > 0.35 is valuable**, and final judging is **70% quantitative, 30% qualitative** (morphology, pipeline and source code). It also requires a 4-page report and reproducible public repository by November 15, 2026. There is no evidence that the 0.55 opaque-submission code trained a better model, and a public score does not guarantee private leaderboard ranking.

References: [IEEE TMI 2025 MaskVSC](https://doi.org/10.1109/TMI.2025.3538336), [MICCAI 2025 GLCP](https://papers.miccai.org/miccai-2025/0376-Paper1868.html), [Skeleton Recall Loss](https://arxiv.org/abs/2404.03010), [official Kaggle competition](https://www.kaggle.com/competitions/filament-segmentation-2026).
